# Steering in LLMs via Activation Engineering

**Goal:** Force an LLM to mention a specific brand name — without prompting — by injecting a steering vector into its internal activations at inference time.

**Model:** Meta Llama 3.1 8B Instruct
**Approach:** Contrastive Activation Addition (CAA) (https://aclanthology.org/2024.acl-long.828.pdf) (example of use in introspection: https://transformer-circuits.pub/2025/introspection/index.html)

**Experiments:**
1. **Baseline** — model generates responses to neutral and brand-relevant prompts (no steering)
2. **Prompted** - a prompted baseline
3. **Steered** — inject a "Coca-Cola" steering vector to bias the model toward mentioning the brand

**Setup:** MacBook with Apple Silicon (MPS backend). Model loaded in float16.


## 0. Installation

Run this cell once to install dependencies.

In [ ]:
# !pip install torch transformers accelerate bitsandbytes sentencepiece protobuf python-dotenv scikit-learn
# NOTE: bitsandbytes 4-bit quantization requires CUDA (Linux/Windows with NVIDIA GPU).
# On MacBook (MPS), you have two options:
#   Option A: Load in float16 (needs ~16GB+ free RAM → works well with 32GB Mac)
#   Option B: Use a smaller model like Llama 3.2 3B in float16 (~6GB)
#
# If you ARE on a CUDA machine, uncomment the line above.
# The notebook detects your platform automatically in the next cell.

## 1. Configuration & Model Loading

In [ ]:
import torch
import json
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from collections import defaultdict
import os
from dotenv import load_dotenv, find_dotenv

# ── CONFIG ──────────────────────────────────────────────────────────
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"  # Requires HF access token if gated

# Brand experiment settings
BRAND = "Coca-Cola"

# Steering settings
STEERING_LAYER = 16          # Which residual stream layer to inject into (we'll sweep this later)
STEERING_SCALE = 4.0         # Multiplier for the steering vector (we'll sweep this too)

# ── DEVICE DETECTION ────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"✅ Using CUDA: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("✅ Using Apple MPS (Metal Performance Shaders)")
else:
    DEVICE = "cpu"
    print("⚠️ Using CPU — this will be slow")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")


# ── LOAD HF TOKEN FROM .env (LOCAL ONLY) ────────────────────────────
dotenv_path = find_dotenv(filename=".env", usecwd=True)
if not dotenv_path:
    raise RuntimeError(
        f".env file not found from current working directory: {os.getcwd()}"
    )

# Important: override=True so a stale empty HF_TOKEN in kernel env gets replaced
load_dotenv(dotenv_path=dotenv_path, override=True)
HF_TOKEN = (os.getenv("HF_TOKEN") or "").strip().strip('"').strip("'")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print(f"✅ Loaded HF_TOKEN from: {dotenv_path}")
else:
    raise RuntimeError(
        f"HF_TOKEN not found/empty in {dotenv_path}. Set: HF_TOKEN=hf_xxx"
    )


# ── LOAD MODEL ──────────────────────────────────────────────────────
# On CUDA: use 4-bit quantization to save VRAM
# On MPS/CPU: load in float16 (no bitsandbytes support)

print(f"Loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

if DEVICE == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    # MPS or CPU — float16
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        token=HF_TOKEN,
        torch_dtype=torch.float16,
        device_map="auto",
    )

model.eval()
print(f"✅ Model loaded. Layers: {model.config.num_hidden_layers}, Hidden size: {model.config.hidden_size}")

# ── Measure activation norms per layer ──────────────────────
# The L2 norm ‖h‖ of the hidden state tells us the magnitude of activations.
# We need this to calibrate steering later: α = fraction × ‖h‖

print("\nMeasuring activation norms across all layers...")
_inputs = tokenizer("The quick brown fox jumps over the lazy dog.",
                    return_tensors="pt", truncation=True, max_length=512)
_inputs = {k: v.to(model.device) for k, v in _inputs.items()}

norms_per_layer = []
for _lid in range(model.config.num_hidden_layers):
    _act = {}
    def _hook(module, input, output, _act=_act):
        h = output[0] if isinstance(output, tuple) else output
        _act['v'] = h.detach()
    _handle = model.model.layers[_lid].register_forward_hook(_hook)
    with torch.no_grad():
        model(**_inputs)
    _handle.remove()
    norms_per_layer.append(
        (_act['v'][0] if _act['v'].dim() == 3 else _act['v']).float().norm(dim=-1).mean().item()
    )

print(f"  Norm range: {min(norms_per_layer):.1f} (layer 0) → "
      f"{max(norms_per_layer):.1f} (layer {norms_per_layer.index(max(norms_per_layer))})")
for i, norm in enumerate(norms_per_layer):
    bar = "█" * int(norm / max(norms_per_layer) * 30)
    print(f"  Layer {i:>2}: ‖h‖={norm:>7.2f}  {bar}")


## 2. Define Steering Prompt Datasets

All prompt sets are defined here in one place. The extraction and PCA cells reference these — no duplicated prompts elsewhere.

- **Generative pairs** — short incomplete sentences where the model is "about to" talk about the brand vs generic topics. Used by the recommended `generative` extraction method and PCA.
- **Direct pairs** — short declarative sentences with/without the brand. Used by the `direct` extraction method.
- **Contrastive chat pairs** — full chat-formatted Q&A with/without the brand in the response. Used by the `mean_diff` and `last_token` methods.

In [ ]:
import prompts as P

# ── CHAT HELPER ──────────────────────────────────────────────────────

def format_chat(user_msg, assistant_msg=None):
    """Format a message using the model's chat template."""
    messages = [{"role": "user", "content": user_msg}]
    if assistant_msg:
        messages.append({"role": "assistant", "content": assistant_msg})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=(assistant_msg is None))



# ═══════════════════════════════════════════════════════════════════════
# CONTRASTIVE PAIR SETS — used by extract_steering_vector()
# Templates loaded from prompts.py; brand substitution happens here.
# ═══════════════════════════════════════════════════════════════════════
def make_generative_pairs(brand_name):
    """Short incomplete sentences for the 'generative' extraction method."""
    return [(pos.format(brand=brand_name), neg.format(brand=brand_name))
            for pos, neg in P.GENERATIVE_PAIRS]

def make_direct_pairs(brand_name):
    """Short declarative sentences for the 'direct' extraction method."""
    return [(pos.format(brand=brand_name), neg.format(brand=brand_name))
            for pos, neg in P.DIRECT_PAIRS]

def make_contrastive_pairs(brand_name):
    """Chat-formatted Q&A pairs for 'mean_diff' and 'last_token' methods."""
    return [
        (format_chat(p["user"], p["brand"].format(brand=brand_name)),
         format_chat(p["user"], p["neutral"]))
        for p in P.CONTRASTIVE_PAIRS
    ]


print(f"✅ Generative pairs: {len(make_generative_pairs('test'))}")
print(f"✅ Direct pairs: {len(make_direct_pairs('test'))}")
print(f"✅ Contrastive chat pairs: {len(make_contrastive_pairs('test'))}")

## 3. Extract Steering Vectors & Test Steering

For each contrastive pair, we run a forward pass and collect the residual stream activations at a target layer. The steering vector is the **mean difference** between positive (brand) and negative (neutral) activations.

We extract at the **last token position** of the prompt (right before generation begins). 

----

Steering uses **Activation Addition (ActAdd)**: at a chosen layer, we add a scaled steering vector to the residual stream during the forward pass:

$$\mathbf{x}' = \mathbf{x} + \alpha \cdot \mathbf{v}$$

where $\mathbf{v}$ is the unit-normalised steering direction and $\alpha$ controls strength. 

The vector $\mathbf{v}$ is extracted from the **last token position** (which carries the most accumulated context), but during generation the hook adds $\alpha \cdot \mathbf{v}$ to **all token positions** in the residual stream. On the first forward pass this nudges the entire prompt representation toward the brand direction, biasing attention patterns and value vectors across all downstream layers. On every subsequent autoregressive step, thanks to KV-cache, only the single newest token passes through the transformer — so the hook effectively steers one token at a time from that point on. The model receives no modified prompt and no system instruction; the intervention is purely geometric and invisible at the text level. 

Rather than setting $\alpha$ as a fixed number, we express it as a **fraction of the activation norm** at the injection layer:

$$\alpha = f \cdot \|\mathbf{h}_l\|$$

where $f$ is the steering fraction (e.g. 0.12) and $\|\mathbf{h}_l\|$ is the **mean L2 norm** of the hidden state at layer $l$, computed as:

$$\|\mathbf{h}_l\| = \frac{1}{T}\sum_{t=1}^{T}\sqrt{\sum_{i=1}^{d} h_{t,i}^2}$$

i.e. the L2 (Euclidean) norm of each token's $d$-dimensional hidden vector, averaged across all $T$ token positions on a calibration sentence (cell 1). This keeps the intervention proportional to the network's own signal magnitude, making it comparable across layers.

We also use **repetition penalty** (1.1) and lower **temperature** (0.6) to keep fluency at higher scales.

---

The cell doesn't do anything but can be run to test the functions.

In [ ]:
def get_residual_activations(text, layer_idx, pooling="last"):
    """
    Run a forward pass and capture the residual stream output at a given layer.
    
    Args:
        text: Input text (already formatted)
        layer_idx: Which transformer layer to hook
        pooling: "last" = last token only, "mean" = mean across all tokens
    
    Returns:
        Activation tensor of shape (hidden_size,)
    """
    activations = {}
    
    def hook_fn(module, input, output):
        if isinstance(output, tuple):
            hidden = output[0]
        elif hasattr(output, 'last_hidden_state'):
            hidden = output.last_hidden_state
        elif hasattr(output, '__getitem__'):
            hidden = output[0]
        else:
            hidden = output
        activations['value'] = hidden.detach()
    
    handle = model.model.layers[layer_idx].register_forward_hook(hook_fn)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    try:
        with torch.no_grad():
            model(**inputs)
    finally:
        handle.remove()
    
    act = activations['value']
    if act.dim() == 3:
        act = act[0]  # (seq_len, hidden_size)
    
    if pooling == "last":
        return act[-1, :].cpu().float()
    elif pooling == "mean":
        return act.mean(dim=0).cpu().float()
    else:
        raise ValueError(f"Unknown pooling: {pooling}")


# ── Pair set & pooling strategy per method ───────────────────────────
EXTRACTION_CONFIG = {
    "generative":  {"pairs_fn": make_generative_pairs,   "pooling": "last"},
    "direct":      {"pairs_fn": make_direct_pairs,       "pooling": "mean"},
    "mean_diff":   {"pairs_fn": make_contrastive_pairs,  "pooling": "mean"},
    "last_token":  {"pairs_fn": make_contrastive_pairs,  "pooling": "last"},
}


def extract_steering_vector(brand_name, layer_idx, method="generative", return_diffs=False):
    """
    Compute the steering vector for a brand at a given layer.
    
    Methods (all use pairs defined in Chapter 2):
        "generative" — (RECOMMENDED) Short incomplete sentences, last-token pooling.
        "direct"     — Short declarative sentences, mean-pooled.
        "mean_diff"  — Contrastive chat pairs, mean-pooled.
        "last_token" — Contrastive chat pairs, last-token pooling.
    """
    config = EXTRACTION_CONFIG[method]
    pairs = config["pairs_fn"](brand_name)
    pooling = config["pooling"]
    
    diffs = []
    for pos_text, neg_text in pairs:
        pos_act = get_residual_activations(pos_text, layer_idx, pooling=pooling)
        neg_act = get_residual_activations(neg_text, layer_idx, pooling=pooling)
        diffs.append(pos_act - neg_act)
    
    steering_vector = torch.stack(diffs).mean(dim=0)
    steering_vector = steering_vector / steering_vector.norm()
    if return_diffs:
        return steering_vector, torch.stack(diffs)
    return steering_vector


def generate_with_steering(prompt, steering_vector=None, scale=STEERING_SCALE, 
                           layer_idx=STEERING_LAYER, max_new_tokens=200):
    """
    Generate a response, optionally injecting a steering vector at a specific layer.
    
    Steering uses Activation Addition: x' = x + α·v
    
    Args:
        prompt: User prompt (will be formatted with chat template)
        steering_vector: The vector to inject (None = no steering, baseline)
        scale: How strongly to steer (higher = stronger effect)
        layer_idx: Which layer to inject into
        max_new_tokens: Generation length
    
    Returns:
        Generated text (assistant response only)
    """
    formatted = format_chat(prompt)
    inputs = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]
    
    handle = None
    if steering_vector is not None:
        sv = steering_vector.to(model.device).to(model.dtype)
        
        def steering_hook(module, input, output):
            def add_steering(hidden_states):
                return hidden_states + scale * sv
            
            if isinstance(output, tuple):
                hidden_states = output[0]
                return (add_steering(hidden_states),) + output[1:]
            elif hasattr(output, 'last_hidden_state'):
                output.last_hidden_state = add_steering(output.last_hidden_state)
                return output
            else:
                return add_steering(output)
        
        handle = model.model.layers[layer_idx].register_forward_hook(steering_hook)
    
    try:
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.6,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1,
            )
    finally:
        if handle is not None:
            handle.remove()
    
    response = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)
    return response.strip()


# ── Extract vectors using the generative method ─────────────────────
print(f"Extracting steering vector (generative method, layer {STEERING_LAYER})...\n")

sv, sv_diffs = extract_steering_vector(BRAND, STEERING_LAYER, method="generative", return_diffs=True)
print(f"  ✅ {BRAND}: shape={sv.shape}, norm={sv.norm():.4f}")

# ── Quick sanity check ──────────────────────────────────────────────
# Tests a few absolute scales to verify steering has an effect.
# The Layer × Scale sweep below will find the calibrated optimum.

test_prompt = "What's a good drink for summer?"

print("=" * 60)
print(f"SANITY CHECK — does steering have any effect?")
print(f"PROMPT: {test_prompt}")
print(f"Layer: {STEERING_LAYER}, method: generative")
print("=" * 60)

for scale_val in [4.0, 8.0, 12.0]:
    print(f"\n🔴 {BRAND} (α={scale_val:.1f}):")
    print(generate_with_steering(test_prompt, steering_vector=sv, scale=scale_val))

## 4. Layer × Scale Sweep

The optimal steering depends on **two coupled parameters**:
- **Layer**: where in the network to inject (different layers encode different levels of abstraction)
- **Scale**: how strongly to steer, expressed as a **fraction of the activation norm** at that layer

A fixed scale like α=8.0 means very different things at different layers — layer 4 has norm ≈ 30, but layer 24 has norm ≈ 72. Expressing scale as a **percentage of the norm** gives a consistent, comparable measure across layers.

This cell runs a **2D grid search** and produces a heatmap to find the sweet spot.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import io as _io, sys as _sys

# ── Tee stdout so we can save a copy to a text file ─────────
_sweep_buf = _io.StringIO()
_orig_stdout = _sys.stdout

class _Tee:
    """Write to both the original stdout and a capture buffer."""
    def __init__(self, orig, buf):
        self._orig, self._buf = orig, buf
    def write(self, data):
        self._orig.write(data)
        self._buf.write(data)
    def flush(self):
        self._orig.flush()

_sys.stdout = _Tee(_orig_stdout, _sweep_buf)


# ── Joint Layer × Scale grid search ─────────────────────────
sweep_layers = [8, 12, 16, 20]
sweep_fracs  = [0.05, 0.10, 0.12, 0.20, 0.30]
sweep_prompts = [
    "What should I drink after a workout?",
    "What are the most recognized brands in the world?",
]

n_total = len(sweep_layers) * len(sweep_fracs) * len(sweep_prompts)
print(f"\nJoint sweep: {len(sweep_layers)} layers × {len(sweep_fracs)} fractions "
      f"× {len(sweep_prompts)} prompts = {n_total} generations")
print(f"Brand: {BRAND} | generative extraction")
print("-" * 70)

SNIPPET_LEN = 250  # max chars to show per response

mention_matrix = np.zeros((len(sweep_layers), len(sweep_fracs)))
coherence_matrix = np.ones((len(sweep_layers), len(sweep_fracs)), dtype=bool)

for i, layer in enumerate(sweep_layers):
    norm = norms_per_layer[layer]
    sv = extract_steering_vector(BRAND, layer, method="generative")
    print(f"\n{'─'*70}")
    print(f"Layer {layer:>2} (‖h‖={norm:.1f})")
    print(f"{'─'*70}")
    
    for j, frac in enumerate(sweep_fracs):
        scale = frac * norm
        hits, coherent = 0, True
        snippets = []
        
        for prompt in sweep_prompts:
            resp = generate_with_steering(prompt, steering_vector=sv, scale=scale,
                                          layer_idx=layer)
            mentioned = BRAND.lower() in resp.lower() or "coke" in resp.lower()
            if mentioned:
                hits += 1
            words = resp.split()
            if len(words) > 5 and max(words.count(w) for w in set(words)) / len(words) > 0.4:
                coherent = False
            tag = "✅" if mentioned else "❌"
            short_p = prompt[:50] + " [...]" if len(prompt) > 50 else prompt
            short_r = resp[:SNIPPET_LEN] + " [...]" if len(resp) > SNIPPET_LEN else resp
            snippets.append((tag, short_p, short_r))
        
        rate = hits / len(sweep_prompts)
        mention_matrix[i, j] = rate
        coherence_matrix[i, j] = coherent
        warn = " ⚠️ incoherent" if not coherent else ""
        header = f"L{layer:>2} - {frac:>4.0%} (α={scale:>5.1f}) → {hits}/{len(sweep_prompts)} mentions{warn} "
        print(f"\n╔{'═'*len(header)}╗")
        print(f"║{header}║")
        print(f"╚{'═'*len(header)}╝")
        for tag, sp, sr in snippets:
            print(f"{tag}\n❓: {sp}")
            print(f"💬: {sr}")
            print(f"{'·'*60}")

# ── Find best (layer, fraction) combo ───────────────────────
# Prefer highest mention rate; penalise incoherent outputs; break ties with lower scale
best_score, best_i, best_j = -1, 0, 0
for i in range(len(sweep_layers)):
    for j in range(len(sweep_fracs)):
        score = mention_matrix[i, j] * (1.0 if coherence_matrix[i, j] else 0.5)
        if score > best_score or (score == best_score and sweep_fracs[j] < sweep_fracs[best_j]):
            best_score, best_i, best_j = score, i, j

BEST_LAYER = sweep_layers[best_i]
BEST_FRAC  = sweep_fracs[best_j]
print(f"\n{'='*70}")
print(f"🏆 BEST: Layer {BEST_LAYER}, {BEST_FRAC:.0%} of norm "
      f"(α={BEST_FRAC * norms_per_layer[BEST_LAYER]:.1f})")
print(f"{'='*70}")

print(f"\n💡 Use BEST_LAYER={BEST_LAYER}, BEST_FRAC={BEST_FRAC:.0%} for the full experiment")

# ── Restore stdout & save sweep log ─────────────────────────────────
_sys.stdout = _orig_stdout
from pathlib import Path as _Path
_sweep_dir = _Path("results") / "_layer_scale"
_sweep_dir.mkdir(parents=True, exist_ok=True)
with open(_sweep_dir / "layer_scale_sweep_output.txt", "w") as _f:
    _f.write(_sweep_buf.getvalue())

# ── Heatmap ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(mention_matrix, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)

for i in range(len(sweep_layers)):
    for j in range(len(sweep_fracs)):
        rate = mention_matrix[i, j]
        label = f"{rate:.0%}"
        if not coherence_matrix[i, j]:
            label += "\n⚠️"
        ax.text(j, i, label, ha="center", va="center",
                fontsize=11, fontweight="bold",
                color="white" if rate > 0.6 else "black")

# Highlight best cell
from matplotlib.patches import Rectangle
ax.add_patch(Rectangle((best_j - 0.5, best_i - 0.5), 1, 1,
                        linewidth=3, edgecolor="#2196F3", facecolor="none"))

ax.set_xticks(range(len(sweep_fracs)))
ax.set_xticklabels([f"{f:.0%}" for f in sweep_fracs])
ax.set_yticks(range(len(sweep_layers)))
ax.set_yticklabels([f"Layer {l}\n(‖h‖={norms_per_layer[l]:.0f})" for l in sweep_layers])
ax.set_xlabel("Steering scale (% of activation norm)", fontsize=12)
ax.set_ylabel("Injection layer", fontsize=12)
ax.set_title(f"Brand Mention Rate — {BRAND}\n"
             f"Generative extraction · ActAdd · {len(sweep_prompts)} prompts/cell",
             fontweight="bold")
plt.savefig(_sweep_dir / "layer_scale_sweep_heatmap.png", dpi=150, bbox_inches="tight")
plt.colorbar(im, ax=ax, label="Mention rate", shrink=0.8)
plt.tight_layout()
print(f"📊 Saved {_sweep_dir / 'layer_scale_sweep_heatmap.png'}")
print(f"💾 Saved {_sweep_dir / 'layer_scale_sweep_output.txt'}")

print(f"💾 Saved layer_scale_sweep_output.txt")
print(f"📊 Saved layer_scale_sweep_heatmap.png")

## 5. Experiment

Configure experiment parameters, load prompts, and define all shared helpers (formatting, saving, charting).

### 5.1 Setup

In [ ]:
# Overriding these variables for the next chapter's 

BEST_LAYER=16
BEST_FRAC= 0.30

In [ ]:
import shutil
import importlib
import json
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import prompts as P

importlib.reload(P)           # always pick up edits to prompts.py

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# ── LOAD EXTERNAL PROMPT DATA ────────────────────────────────────────
NEUTRAL_PROMPTS  = P.NEUTRAL_PROMPTS
STIMULUS_PROMPTS = P.STIMULUS_PROMPTS

print(f"✅ Neutral test prompts: {len(NEUTRAL_PROMPTS)}")
print(f"✅ Stimulus test prompts: {len(STIMULUS_PROMPTS)}")

# ── GLOBAL RESULTS STORE ────────────────────────────────────────────
all_results = {}   # condition → list[dict]

PROMPT_TYPES = ["neutral", "stimulus"]


# ═══════════════════════════════════════════════════════════════════════
# FORMATTING HELPERS
# ═══════════════════════════════════════════════════════════════════════
CONDITION_COLORS = {
    "baseline": "#a8dadc",
    "prompted": "#f4a261",
    "steered":  "#e63946",
}

CONDITION_LABELS = {
    "baseline": "Baseline",
    "prompted": f"Prompted ({BRAND})",
    "steered":  f"Steered ({BRAND})",
}


def format_result(r, index, total, max_len=None):
    """Format one result entry. Truncates prompt/response when max_len is set."""
    mention_tag = "✅" if r["mentions_brand"] else "❌"
    type_tag = "⬜️" if r['prompt_type'] == "neutral" else "🟨"
    condition_tag = {"steered": "▶️", "prompted": "⏩️", "baseline": "⏹️"}.get(r['condition'], "⏹️")
    prompt_text  = r["prompt"]
    response_text = r["response"]
    if max_len is not None:
        prompt_text   = prompt_text[:80]  + " [...]" if len(prompt_text)  > 80  else prompt_text
        response_text = response_text[:max_len] + " [...]" if len(response_text) > max_len else response_text
    lines = [
        f"{mention_tag} [{index}/{total}] | {condition_tag} {r['condition']} | {type_tag} {r['prompt_type']} | mentions {BRAND}: {'YES' if r['mentions_brand'] else 'no'}",
        f"❓ {prompt_text}",
        f"💬 {response_text}",
        "─" * 60,
    ]
    return "\n".join(lines)


def section_header(title, style="major"):
    """Return a framed section header string."""
    if style == "major":
        w = len(title) + 4
        return (f"\n╔{'═' * w}╗\n"
                f"║  {title}  ║\n"
                f"╚{'═' * w}╝")
    else:
        w = len(title) + 2
        return (f"\n┌{'─' * w}┐\n"
                f"│ {title} │\n"
                f"└{'─' * w}┘")


def compute_mention_rate(results, prompt_type, condition):
    subset = [r for r in results if r["prompt_type"] == prompt_type and r["condition"] == condition]
    if not subset:
        return 0.0
    return sum(1 for r in subset if r["mentions_brand"]) / len(subset)


# ═══════════════════════════════════════════════════════════════════════
# SAVE HELPERS
# ═══════════════════════════════════════════════════════════════════════
def make_output_dir(name):
    """Create/clear an output directory under RESULTS_DIR.
    For shared directories (starting with '_'), refuse to overwrite
    automatically — the user must delete them manually first.
    For layer/frac directories, auto-clear as before.
    """
    d = RESULTS_DIR / name
    if d.exists():
        if name.startswith("_"):
            raise FileExistsError(
                f"🛑 Directory already exists: {d}\n"
                f"   Baseline/prompted results are expensive to regenerate.\n"
                f"   Delete it manually if you really want to re-run:\n"
                f"   !rm -rf {d}"
            )
        shutil.rmtree(d)
    d.mkdir(parents=True)
    return d


def save_summary(results, conditions, out_dir,
                 subtitle="", filename="summary.txt"):
    """Write summary with mention-rate table. Returns summary_data dict."""
    summary_data = {}
    lines = ["=" * 60, f"BRAND MENTION RATES — {BRAND}"]
    if subtitle:
        lines.append(subtitle)
    lines.append("=" * 60)
    lines.append(f"{'Condition':<15} {'Prompt Type':<12} {'Mention Rate'}")
    lines.append("-" * 45)
    for cond in conditions:
        for ptype in PROMPT_TYPES:
            rate = compute_mention_rate(results, ptype, cond)
            lines.append(f"{cond:<15} {ptype:<12} {rate:>10.0%}")
            summary_data[(cond, ptype)] = rate
    text = "\n".join(lines)
    (out_dir / filename).write_text(text)
    print(f"\n{text}")
    print(f"\n💾 Saved {out_dir / filename}")
    return summary_data


def save_outputs(results, conditions, out_dir, filename="outputs.txt"):
    """Write full prompt/response traces."""
    lines = []
    for cond in conditions:
        for ptype in PROMPT_TYPES:
            subset = [r for r in results
                      if r["condition"] == cond and r["prompt_type"] == ptype]
            if not subset:
                continue
            lines.append(section_header(f"{cond} · {ptype}", style="major"))
            for idx, r in enumerate(subset, 1):
                lines.append(format_result(r, idx, len(subset)))
    (out_dir / filename).write_text("\n".join(lines))
    print(f"💾 Saved {out_dir / filename}")


def save_json(results, out_dir, filename="results.json", extra_meta=None):
    """Write results JSON."""
    payload = {"brand": BRAND, "results": results}
    if extra_meta:
        payload.update(extra_meta)
    (out_dir / filename).write_text(json.dumps(payload, indent=2, default=str))
    print(f"💾 Saved {out_dir / filename}")


def save_bar_chart(summary_data, conditions, out_dir,
                   filename="bar_chart.png", title_extra=""):
    """Save a grouped bar chart for one or more conditions."""
    labels = [f"Neutral\nPrompts ({len(NEUTRAL_PROMPTS)})",
              f"Stimulus\nPrompts ({len(STIMULUS_PROMPTS)})"]
    x = np.arange(len(labels))
    n = len(conditions)
    width = 0.7 / n

    fig, ax = plt.subplots(figsize=(max(6, 2 + 3 * n), 5))
    all_bars = []
    for k, cond in enumerate(conditions):
        rates = [summary_data.get((cond, pt), 0) for pt in PROMPT_TYPES]
        offset = (k - (n - 1) / 2) * width
        bars = ax.bar(x + offset, rates, width,
                      label=CONDITION_LABELS.get(cond, cond),
                      color=CONDITION_COLORS.get(cond, "#999999"))
        all_bars.extend(bars)

    ax.set_ylabel("Mention Rate")
    title = f"Brand Mention Rate — {BRAND}"
    if title_extra:
        title += f"\n{title_extra}"
    ax.set_title(title, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.1)
    ax.legend()

    for bar in all_bars:
        h = bar.get_height()
        ax.annotate(f"{h:.0%}", xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", fontsize=11)

    plt.tight_layout()
    path = out_dir / filename
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"📊 Saved {path}")


# ═══════════════════════════════════════════════════════════════════════
# RUN HELPERS
# ═══════════════════════════════════════════════════════════════════════
def run_experiment(prompts, prompt_type, steering_vector=None,
                   scale=None, layer_idx=None):
    """Run prompts through generate_with_steering. Returns list of result dicts."""
    if scale is None:
        scale = exp_scale
    if layer_idx is None:
        layer_idx = exp_layer

    results = []
    condition = "baseline" if steering_vector is None else "steered"

    for i, prompt in enumerate(prompts):
        response = generate_with_steering(prompt, steering_vector=steering_vector,
                                          scale=scale, layer_idx=layer_idx)
        mentioned = BRAND.lower() in response.lower() or "coke" in response.lower()
        r = {
            "prompt_type": prompt_type,
            "condition": condition,
            "brand": BRAND,
            "layer": layer_idx,
            "scale": scale if steering_vector is not None else 0,
            "prompt": prompt,
            "response": response,
            "mentions_brand": mentioned,
        }
        results.append(r)
        print(format_result(r, i + 1, len(prompts), max_len=250))

    return results


SYSTEM_INSTRUCTION = f"Mention {BRAND} in your responses when relevant, the mention should feel natural and not forced. Brand: {BRAND}."


def generate_with_system_prompt(prompt, max_new_tokens=200):
    """Generate using a system prompt that instructs the model to mention a brand."""
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": prompt},
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False,
                                              add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.6,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
    return tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()


def run_prompted_experiment(prompts, prompt_type):
    """Run prompts with system-prompt approach. Returns list of result dicts."""
    results = []
    for i, prompt in enumerate(prompts):
        response = generate_with_system_prompt(prompt)
        mentioned = BRAND.lower() in response.lower() or "coke" in response.lower()
        r = {
            "prompt_type": prompt_type,
            "condition": "prompted",
            "brand": BRAND,
            "layer": None,
            "scale": 0,
            "prompt": prompt,
            "response": response,
            "mentions_brand": mentioned,
        }
        results.append(r)
        print(format_result(r, i + 1, len(prompts), max_len=250))
    return results


# ── CONFIGURE — use best parameters from the joint sweep ─────────────
try:
    exp_layer = BEST_LAYER
    exp_frac  = BEST_FRAC
    exp_scale = BEST_FRAC * norms_per_layer[BEST_LAYER]
except NameError:
    exp_layer = STEERING_LAYER
    exp_frac  = 0.12
    exp_scale = 8.0
    print("⚠️ Run the Layer × Scale sweep first! Using fallback defaults.")

print(f"\n⚙️  Steering config: Layer {exp_layer} | Scale {exp_scale:.1f} ({exp_frac:.0%} of norm)")
print(f"✅ Ready to run experiments")

### 5.2 Baseline

Generate responses to all prompts with **no intervention** — no steering vector, no system prompt. This establishes the control condition.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# RUN — baseline (no steering, no system prompt)
# ═══════════════════════════════════════════════════════════════════════
baseline_dir = make_output_dir("_baseline")
print(f"📁 Output directory: {baseline_dir}\n")

baseline_results = []

print(section_header("BASELINE (no steering)", style="major"))

print(section_header("Neutral prompts (baseline)", style="minor"))
baseline_results.extend(run_experiment(NEUTRAL_PROMPTS, "neutral"))

print(section_header("Stimulus prompts (baseline)", style="minor"))
baseline_results.extend(run_experiment(STIMULUS_PROMPTS, "stimulus"))

all_results["baseline"] = baseline_results
print(f"\n✅ Baseline results: {len(baseline_results)}")

# ── SAVE ─────────────────────────────────────────────────────────────
sd = save_summary(baseline_results, ["baseline"], baseline_dir)
save_outputs(baseline_results, ["baseline"], baseline_dir)
save_bar_chart(sd, ["baseline"], baseline_dir)
save_json(baseline_results, baseline_dir)

print(f"\n📁 All baseline outputs in: {baseline_dir}")

### 5.3 Prompt-Based Steering

What if we steer with prompts? 

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# RUN — prompted (system instruction, no activation steering)
# ═══════════════════════════════════════════════════════════════════════
prompted_dir = make_output_dir("_prompted")
print(f"📁 Output directory: {prompted_dir}")
print(f"System instruction: \"{SYSTEM_INSTRUCTION}\"\n")

prompted_results = []

print(section_header(f"PROMPTED — system prompt ({BRAND})", style="major"))

print(section_header("Neutral prompts (prompted)", style="minor"))
prompted_results.extend(run_prompted_experiment(NEUTRAL_PROMPTS, "neutral"))

print(section_header("Stimulus prompts (prompted)", style="minor"))
prompted_results.extend(run_prompted_experiment(STIMULUS_PROMPTS, "stimulus"))

all_results["prompted"] = prompted_results
print(f"\n✅ Prompted results: {len(prompted_results)}")

# ── SAVE ─────────────────────────────────────────────────────────────
sd = save_summary(prompted_results, ["prompted"], prompted_dir,
                  subtitle=f'System instruction: "{SYSTEM_INSTRUCTION}"')
save_outputs(prompted_results, ["prompted"], prompted_dir)
save_bar_chart(sd, ["prompted"], prompted_dir,
               title_extra=f'System prompt: "{SYSTEM_INSTRUCTION}"')
save_json(prompted_results, prompted_dir,
          extra_meta={"system_instruction": SYSTEM_INSTRUCTION})

print(f"\n📁 All prompted outputs in: {prompted_dir}")

### 5.4. Activation Steering

Inject the steering vector into the residual stream via **ActAdd**. This is the invisible intervention — no change to the prompt, no system instruction. The model doesn't "know" it's being steered.

After running, we also produce a **cross-condition comparison** (baseline vs prompted vs steered).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# RUN — activation steering (ActAdd)
# ═══════════════════════════════════════════════════════════════════════
frac_str = f"{exp_frac:.2f}".replace(".", "")       # 0.12 → "012"
steered_dir = make_output_dir(f"L{exp_layer}_F{frac_str}")
print(f"📁 Output directory: {steered_dir}")
print(f"Layer: {exp_layer} | Scale: {exp_scale:.1f} ({exp_frac:.0%} of norm)")

# Extract steering vector at the optimal layer
print(f"Extracting vector at layer {exp_layer}...")
sv_exp = extract_steering_vector(BRAND, exp_layer, method="generative")

steered_results = []

print(section_header(f"STEERED — {BRAND} (α={exp_scale:.1f})", style="major"))

print(section_header("Neutral prompts (steered)", style="minor"))
steered_results.extend(run_experiment(NEUTRAL_PROMPTS, "neutral", sv_exp))

print(section_header("Stimulus prompts (steered)", style="minor"))
steered_results.extend(run_experiment(STIMULUS_PROMPTS, "stimulus", sv_exp))

all_results["steered"] = steered_results
print(f"\n✅ Steered results: {len(steered_results)}")

# ── SAVE — steered condition ─────────────────────────────────────────
sd = save_summary(steered_results, ["steered"], steered_dir,
                  subtitle=f"Layer {exp_layer} | α={exp_scale:.1f} ({exp_frac:.0%} of norm)")
save_outputs(steered_results, ["steered"], steered_dir)
save_bar_chart(sd, ["steered"], steered_dir,
               title_extra=f"Layer {exp_layer}, α={exp_scale:.1f} ({exp_frac:.0%} norm)")
save_json(steered_results, steered_dir,
          extra_meta={"layer": exp_layer, "scale": exp_scale, "frac": exp_frac})


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# COMPARISON — load all three conditions from saved JSONs
# ═══════════════════════════════════════════════════════════════════════

# Locate the steered results directory from current config
frac_str = f"{exp_frac:.2f}".replace(".", "")
steered_json_path = RESULTS_DIR / f"L{exp_layer}_F{frac_str}" / "results.json"
baseline_json_path = RESULTS_DIR / "_baseline" / "results.json"
prompted_json_path = RESULTS_DIR / "_prompted" / "results.json"

for p in [baseline_json_path, prompted_json_path, steered_json_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing: {p}  — run the corresponding experiment cell first")

baseline_data = json.loads(baseline_json_path.read_text())
prompted_data = json.loads(prompted_json_path.read_text())
steered_data  = json.loads(steered_json_path.read_text())

# Extract steering metadata
steer_layer = steered_data.get("layer", exp_layer)
steer_scale = steered_data.get("scale", exp_scale)
steer_frac  = steered_data.get("frac", exp_frac)

combined = baseline_data["results"] + prompted_data["results"] + steered_data["results"]
comp_dir = steered_json_path.parent   # save comparison files alongside the steered run

print(f"Loaded {len(baseline_data['results'])} baseline + "
      f"{len(prompted_data['results'])} prompted + "
      f"{len(steered_data['results'])} steered results from JSON")
print(f"Steered config: Layer {steer_layer} | α={steer_scale:.1f} ({steer_frac:.0%} of norm)")

print(section_header("COMPARISON — all conditions", style="major"))

comp_subtitle = (f"All conditions comparison — "
                 f"Steered: L{steer_layer} α={steer_scale:.1f} ({steer_frac:.0%} norm)")

comp_sd = save_summary(combined, ["baseline", "prompted", "steered"],
                       comp_dir,
                       subtitle=comp_subtitle,
                       filename="summary_comparison.txt")

save_bar_chart(comp_sd, ["baseline", "prompted", "steered"],
               comp_dir,
               filename="comparison_chart.png",
               title_extra=f"Steered: Layer {steer_layer}, α={steer_scale:.1f} ({steer_frac:.0%} norm)")

print(f"\n📁 Comparison outputs in: {comp_dir}")